# Export check: does the served adapter answer like the notebook?

The fine-tune is trained and judged in Transformers, but in production it is served by vLLM as a LoRA module.
If the two disagree in kind (one reply runs to the token cap while the other stops, or one is twice as long),
the cause is almost always the chat template or the end-of-turn token, and every serving result would be wrong.

This notebook sends the same 10 fixed prompts (`--set export`: the first five validation and the first five
held-out-intent prompts), greedily, through:

1. **vLLM** on GPU 1: the adapter as a LoRA module, and the base model alone;
2. **Transformers** on GPU 0: the adapter merged into the base, and the base model alone.

The base model is the control: fp16 kernels differ between the two stacks, so some wording drift is normal.
If the base drifts as much as the adapter does, the drift comes from the stack, not the export. The Mac replies
already in the repo (`evals/style/*-mac-export.jsonl`) are a third reference.

**Pass:** no prompt in the adapter comparison "differs in kind". Wording differences alone are fine.

**Before running (Kaggle):** GPU **T4 x2**, Internet **On**. The `HF_TOKEN` secret is optional (both models are
public). **Run All** takes about 20–25 minutes, most of it installing and starting vLLM.

**After:** the last cell writes **`sft-export-check.zip`** to `/kaggle/working/` and prints a download link.
Download it **before stopping the session**: an interactive session's files are gone once it stops.

In [ ]:
# 1. Settings.
ADAPTER = "123jagjot/sft1-r16-lr2e-4-bitext_v1"      # the adapter on the Hub
LORA_NAME = "nova-sft1"                              # the model name vLLM serves it under
MODEL = "Qwen/Qwen3.5-2B"
OUT = "evals/style"
URL = "http://localhost:8000/v1"

import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN secret: fine, both models are public")

In [ ]:
# 2. Clone the repo; install the training stack (for the Transformers side) and the OpenAI client (for vLLM).
import subprocess, sys
REPO_DIR = os.path.abspath("customer-support-SLM")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
%pip install -q -r requirements-gpu.txt
%pip install -q openai
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. vLLM in its own environment, pinned to the versions that passed the risk tests.
!uv --version || pip install -q uv
!uv venv -q /tmp/vllm-env --python 3.12
!uv pip install -q --python /tmp/vllm-env/bin/python -r requirements-serve.txt

In [ ]:
# 4. Download the adapter, then start vLLM on GPU 1 with the base model and the adapter as a LoRA module.
import json, time, urllib.request
from huggingface_hub import snapshot_download
ADAPTER_DIR = snapshot_download(ADAPTER, local_dir="/kaggle/working/adapter",
                                allow_patterns=["adapter_model.safetensors", "adapter_config.json"])
RANK = json.load(open(f"{ADAPTER_DIR}/adapter_config.json"))["r"]
FLAGS = ("--dtype half --max-model-len 8192 --language-model-only "
         "--enable-auto-tool-choice --tool-call-parser qwen3_coder --port 8000 "
         f"--enable-lora --lora-modules {LORA_NAME}={ADAPTER_DIR} --max-lora-rank {max(RANK, 16)}")
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(["bash", "-c", f"source serving/t4_libcuda_fix.sh && "
                           f"exec /tmp/vllm-env/bin/vllm serve {MODEL} {FLAGS}"],
                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                          env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"))
t0 = time.time()
while True:
    try:
        served = json.load(urllib.request.urlopen(f"{URL}/models", timeout=5))
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1500:
            print(open("/kaggle/working/vllm.log").read()[-4000:])
            raise RuntimeError("vLLM did not start; full log in /kaggle/working/vllm.log")
        time.sleep(10)
print(f"vLLM up after {time.time() - t0:.0f}s, rank {RANK}, serving", [m["id"] for m in served["data"]])

In [ ]:
# 5. The 10 export prompts through vLLM: the adapter, then the base model. Then stop the server.
!python evals/generate_replies.py generate --base-url {URL} --model {LORA_NAME} --set export --out {OUT}/sft1-vllm-export.jsonl
!python evals/generate_replies.py generate --base-url {URL} --model {MODEL} --set export --out {OUT}/base-vllm-export.jsonl
server.terminate()

In [ ]:
# 6. The same prompts through Transformers on GPU 0: the adapter merged into the base, then the base alone.
!CUDA_VISIBLE_DEVICES=0 python evals/generate_replies.py generate --provider local --model {MODEL} --adapter {ADAPTER_DIR} --set export --out {OUT}/sft1-notebook-export.jsonl
!CUDA_VISIBLE_DEVICES=0 python evals/generate_replies.py generate --provider local --model {MODEL} --set export --out {OUT}/base-notebook-export.jsonl

In [ ]:
# 7. Compare. The adapter comparison is the check; the base comparison is the control for stack drift.
!python evals/generate_replies.py compare {OUT}/sft1-notebook-export.jsonl {OUT}/sft1-vllm-export.jsonl | tee {OUT}/export-check-sft1.txt
!python evals/generate_replies.py compare {OUT}/base-notebook-export.jsonl {OUT}/base-vllm-export.jsonl | tee {OUT}/export-check-base.txt
print("\nThird reference, the Mac (same weights, MPS):")
!python evals/generate_replies.py compare {OUT}/sft1-mac-export.jsonl {OUT}/sft1-vllm-export.jsonl | tail -1
!python evals/generate_replies.py compare {OUT}/base-mac-export.jsonl {OUT}/base-vllm-export.jsonl | tail -1
for name in ["sft1", "base"]:
    print(f"{name}: {open(f'{OUT}/export-check-{name}.txt').read().strip().splitlines()[-1]}")

In [ ]:
# 8. Package the results and print a download link. Download before stopping the session.
from IPython.display import FileLink
!cd {OUT} && zip -q /kaggle/working/sft-export-check.zip *-vllm-export.jsonl *-notebook-export.jsonl export-check-*.txt
!cp /kaggle/working/vllm.log /tmp/vllm.log && cd /tmp && zip -q /kaggle/working/sft-export-check.zip vllm.log
!unzip -l /kaggle/working/sft-export-check.zip
os.chdir("/kaggle/working")
FileLink("sft-export-check.zip")